# [실습] LangChain Tool Call

LLM은 언어 모델이지만, 특수한 형식의 출력을 통해 외부 도구와 소통할 수 있습니다.   

1. LLM에게 프롬프트를 통해 도구의 사용법을 전달합니다.   
2. LLM은 도구가 필요한 경우 파싱 가능한 출력을 생성합니다.   
3. 해당 출력을 파싱하여 실제 도구를 실행하는 과정을 자동화합니다.
4. 실행 결과를 다시 LLM에 전달하면, LLM은 결과를 참고하여 다음 출력을 생성합니다.


# 라이브러리 설정 및 LLM 불러오기

In [ ]:
!pip install langchain==0.3.27 langchain_community==0.3.27 langgraph==0.6.8 langchain_google_genai langchain_openai langchain_community langchain_tavily -q

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
try:
    llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3)
    llm_nonreasoning = ChatOpenAI(model='gpt-4.1-mini', temperature=0.3)
    print("✅ GPT API 사용 가능!")
except:
    print("❌ GPT API 사용 불가- API 키를 확인하세요!")
try:
    llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, 
                                rate_limiter=rate_limiter)
    llm_gemini_nonreasoning = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.3, 
                                rate_limiter=rate_limiter)
    print("✅ Gemini API 사용 가능!")
except:
    print("❌ Gemini API 사용 불가- API 키를 확인하세요!")

✅ GPT API 사용 가능!
✅ Gemini API 사용 가능!


In [4]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print("필수 모듈 임포트 완료")

필수 모듈 임포트 완료


## LLM에게 어려운 질문하기   

LLM에게 학습하지 못한 범위의 질문을 전달하면 정확도가 떨어지거나 잘못된 답변을 생성합니다.   

In [ ]:
llm.invoke("케이팝 데몬헌터스가 뭐야?")
# 5-mini: 24.6 까지만 

AIMessage(content='좋은 질문이에요 — 문맥이 조금 더 필요해요. “케이팝 데몬헌터스”를 어디서 보셨나요?(예: 트위터, 유튜브, 웹툰, 롭록스 게임, 팬픽 등)\n\n가능한 해석 몇 가지를 짧게 알려드릴게요:\n- 팬픽/팬아트 장르: 케이팝 아이돌들을 악마 사냥꾼으로 등장시키는 팬 창작물일 수 있어요.\n- 특정 프로젝트/그룹명: 유튜브 채널, 웹툰, 게임(예: 팬 제작 게임)이나 롤플레이 그룹 이름일 가능성도 있습니다.\n- 밈·서브컬처 용어: 팬덤 내에서 쓰는 별명이나 패러디 제목일 수도 있어요.\n\n어떤 걸 원하세요? (예: 그 작품의 줄거리 요약, 어디서 보는지, 누가 만들었는지 등) 더 자세한 출처나 링크를 알려주시면 정확히 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 685, 'prompt_tokens': 19, 'total_tokens': 704, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C7xt9K5XfOJzhhCc3PSxguvT5f7Og', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--adc39cb0-cabb-4d5c-8803-a75ea8750a14-0', usage_metadata={'input_tokens': 19, '

In [6]:
llm_gemini.invoke("케이팝 데몬헌터스가 뭐야?")

AIMessage(content='"케이팝 데몬헌터스(K-Pop Demon Hunters)"는 특정 작품 하나의 이름이라기보다는, **케이팝 아이돌의 화려함과 퍼포먼스에 악마나 초자연적인 존재를 사냥하는 \'데몬 헌터\'의 액션/판타지 요소를 결합한 장르나 콘셉트**를 의미합니다.\n\n즉, 무대 위에서는 빛나는 아이돌이지만, 실제로는 비밀리에 초자연적인 위협으로부터 세상을 지키는 능력을 가진 존재라는 설정이죠.\n\n주로 다음과 같은 형태로 나타납니다:\n\n1.  **웹툰/웹소설:**\n    *   가장 활발하게 볼 수 있는 형태입니다. 많은 K-Pop 기획사들이 소속 아이돌 그룹의 세계관(Universe)을 확장하기 위해 웹툰이나 웹소설을 제작하는데, 이 과정에서 아이돌 멤버들이 판타지 세계의 영웅이나 데몬 헌터로 등장하는 경우가 많습니다.\n    *   **예시:**\n        *   **ENHYPEN(엔하이픈)의 \'DARK MOON: 달의 제단\':** 뱀파이어 소년들과 늑대인간들, 그리고 특별한 능력을 가진 소녀의 이야기를 다룬 판타지 웹툰/웹소설입니다.\n        *   **TXT(투모로우바이투게더)의 \'별을 쫓는 소년들\':** 마법 능력을 가진 아이돌 멤버들이 세상을 구하는 이야기를 다룹니다.\n        *   **BTS(방탄소년단)의 \'7FATES: CHAKHO\':** 한국 전통 설화 속 범 사냥꾼(착호갑사)을 모티브로 한 도시 판타지 웹툰/웹소설입니다.\n\n2.  **게임:**\n    *   간혹 인디 게임이나 팬들이 만든 모드(Mod) 등에서 K-Pop 아이돌의 이미지를 차용하거나, 아예 K-Pop 아이돌을 모티브로 한 데몬 헌터 캐릭터가 등장하는 경우가 있습니다.\n    *   **예시:** \'Project: Bellatores\'와 같이 K-Pop 요소를 넣은 데몬 헌팅 게임 프로젝트가 개발되기도 합니다.\n\n3.  **팬픽션/AU (Alternate Universe):**\n    *   팬덤 내에

Anthropic의 기술 블로그에서는, LLM의 외연을 확장하는 방법으로 검색(Retrieval)과 도구(Tool)을 제안했습니다.   
Retrieval은 이후의 RAG에서 자세히 다루도록 하고, 이번 과정에서는 Tool에 대해 이해해 보겠습니다.

## Tool

Tool은 LLM이 이해하고 사용할 수 있는 도구입니다.   
프롬프트를 통해 설명을 전달하므로, 넓은 의미에서 Context Engineering에 해당합니다.   


In [7]:
# OpenAI API의 도구 생성 기능

from openai import pydantic_function_tool
from pydantic import BaseModel, Field

class web_search(BaseModel):
    """웹 검색을 수행합니다."""
    query: str= Field(description="""검색 키워드""")

web_search_tool = pydantic_function_tool(web_search)

web_search_tool

{'type': 'function',
 'function': {'name': 'web_search',
  'strict': True,
  'parameters': {'description': '웹 검색을 수행합니다.',
   'properties': {'query': {'description': '검색 키워드',
     'title': 'Query',
     'type': 'string'}},
   'required': ['query'],
   'title': 'web_search',
   'type': 'object',
   'additionalProperties': False},
  'description': '웹 검색을 수행합니다.'}}

위 결과를 Tool Schema라고 하며, 해당 내용이 LLM의 프롬프트에 포함되어 툴 사용 정보를 전달합니다.

## LangChain Tool 설정하기

LangChain은 Tool Calling을 쉽게 연동하기 위한 기능을 제공합니다.   


LangChain에서 자체적으로 지원하는 Tool을 사용하거나, RAG의 Retriever를 Tool로 변환하는 것도 가능합니다.     
또한, 함수를 Tool로 변환할 수도 있습니다.

가장 대표적인 built-in 툴인 Tavily Search (http://app.tavily.com/) 를 연결해 보겠습니다.

해당 도메인에 접속하여, 구글 계정으로 회원가입을 진행해 주세요.

이후, API 키를 생성하여 env 파일에 `TAVILY_API_KEY`로 추가합니다.

In [9]:
from langchain_tavily import TavilySearch

# 파일 업데이트 후 실행
load_dotenv(override=True)

tavily_search = TavilySearch(
    max_results=5,
    # 더 많은 옵션은 Tavily API 문서 Playground 참고
    )
tavily_search

TavilySearch(max_results=5, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))

In [10]:
search_docs = tavily_search.invoke("케이팝 데몬 헌터스의 의미")
search_docs

{'query': '케이팝 데몬 헌터스의 의미',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://blog.naver.com/influencer33/223968960312?fromRss=true&trackingCode=rss',
   'title': '케이팝 데몬 헌터스(K-pop Demon Hunters) 혼문 뜻 : 네이버 블로그',
   'content': '이 작품은 단순한 액션 판타지가 아니라, 음악의 진심이 사람들의 영혼을 울리고, 그 울림이 세상을 지킨다는 상징을 담고 있습니다. 혼문은 곧 존중과',
   'score': 0.8036831,
   'raw_content': None},
  {'url': 'https://brunch.co.kr/@dies-imperi/900',
   'title': '케이팝 데몬 헌터스 영화와 OST 해설 - 브런치',
   'content': '케이팝 데몬 헌터스 (KPop Demon Hunters·2025)》 | 《케이팝 데몬 헌터스》은 K-Pop 팬이든 아니든 즐겁게 해준다. 악령을 사냥하는 걸그룹을 주인공 삼아 퇴마물의',
   'score': 0.68140936,
   'raw_content': None},
  {'url': 'http://blog.naver.com/tastefood_in_korea/223912221489',
   'title': '케이팝 데몬 헌터스 Kpop demon hunters- golden 가사해석 발음',
   'content': '<케이팝 데몬 헌터스>는 단순한 애니메이션이 아닙니다. K-팝의 음악, 안무, 서사 구조, 캐릭터 설정까지 실제 아이돌 생태계의 모든 요소를 세심하게 담',
   'score': 0.6538572,
   'raw_content': None},
  {'url': 'https://woman.chosun.com/news/articleView.ht

API 호출은 대부분 사용자가 원하는 정보보다 많은 내용을 전달합니다.   
불필요한 내용을 제거하는 함수를 구성합니다.

In [14]:
from langchain_core.tools import tool

@tool
def tavily_search(query, max_results = 5):
    """Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.
query는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.
사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요."""
    tavily_search = TavilySearch(max_results=max_results)
    results = tavily_search.invoke(query)['results']

    context = ''
    for result in results:
        # result: dict
        doc_content = 'URL: '+ result.get('url') +'\n Title:'+ result.get('title') +'\n Content:' + result.get('content')
        context += doc_content + '\n\n---\n\n'

    return context

print(tavily_search.invoke({'query':'케이팝 데몬헌터스'}))





URL: https://ko.wikipedia.org/wiki/%EC%BC%80%EC%9D%B4%ED%8C%9D_%EB%8D%B0%EB%AA%AC_%ED%97%8C%ED%84%B0%EC%8A%A4
 Title:케이팝 데몬 헌터스 - 위키백과, 우리 모두의 백과사전
 Content:**《케이팝 데몬 헌터스》**(영어: KPop Demon Hunters)는 2025년 공개된 미국의 뮤지컬, 판타지, 코미디 애니메이션 영화이다. 2021년 3월, 소니 픽처스 애니메이션에서 《K-Pop: Demon Hunter》 라는 제목의 영화가 제작 중이라고 발표되었다. 유는 이 영화가 소니 픽처스 애니메이션의 "최근 히트작인 스파이더버스 프랜차이즈"와 "일종의 계보"를 공유하며, "유사한 시각적 스타일"을 공유하지만, "대부분은 영화적 측면에서 더 총체적이고 기술적인 감각을 빌려온다"고 언급하며, "유려한" 액션, "인상적인" 미술, 그리고 "역동적인 스토리텔링 도구" 역할을 하는 음악을 강조했다.Io9의 이사야 콜베르트도 유사하게 "소니 픽처스 애니메이션의 애니메이션 팀이 아낌없이 또 다른 시각적 즐거움을 제공하며 화려하고 생동감 있는 애니메이션을 선보였다"고 평하며, 스파이더버스 성공 이후의 성과를 언급했다. 골드버그는 매기 강과 크리스 아펠한스 감독이 "설정이 기이할 수 있음에도 불구하고, 판돈을 실제처럼 다루어야 한다는 것을 이해하고 있다"는 점을 칭찬했다.콜라이더&action=edit&redlink=1 "Collider (웹사이트) (없는 문서)")의 제프 이윙은 이 영화가 "아름답게 미친 판타지 전제"를 성공적으로 구현했으며, "악마, 음악, 사냥꾼을 둘러싼 흥미로운 전설을 가지고 있는데, 이는 새롭지만 풍부하게 느껴진다"고 평했다.

---

URL: https://www.hankyung.com/article/2025080492127
 Title:케이팝 데몬 헌터스 대박 나더니…넷플릭스 이것까지 만든다 - 한국경제
 Content:'케이팝 데몬 헌터스'는 K팝 아이

위에서 만든 함수에, @tool decorator를 붙이면 랭체인의 툴로 변환됩니다.

## LLM에 Tool 연결하기   

생성한 툴은 llm.bind_tools()를 통해 LLM에 연결할 수 있습니다.    

In [17]:
tools = [tavily_search]

llm_with_tools = llm.bind_tools(tools)
llm_with_tools

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001D047EF1BE0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D047EF2660>, root_client=<openai.OpenAI object at 0x000001D047D23230>, root_async_client=<openai.AsyncOpenAI object at 0x000001D047EF23C0>, model_name='gpt-5-mini', model_kwargs={}, openai_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요.', 'parameters': {'properties': {'query': {}, 'max_results': {'default': 5}}, 'required': ['query'], 'type': 'object'}}}]}, config={}, config_factories=[])

랭체인에서, tool 정보는 tools에 저장되는데요.   
해당 내용은 랭체인 내부에서 json Schema 형식으로 프롬프트에 포함됩니다.

In [20]:
llm_with_tools.kwargs['tools']

[{'type': 'function',
  'function': {'name': 'tavily_search',
   'description': 'Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요.',
   'parameters': {'properties': {'query': {}, 'max_results': {'default': 5}},
    'required': ['query'],
    'type': 'object'}}}]

LLM은 프롬프트로 주어지는 툴 정보를 바탕으로 Tool의 사용을 결정합니다.    
Schema를 통해, 툴의 의미와 사용 방법, 형식을 이해합니다.

Tool을 탑재한 LLM을 실행해 보겠습니다.

In [21]:
prompt = ChatPromptTemplate(
    [
        ('system', '주어진 검색 툴을 사용해서 질문에 답변하세요.'),
        ('human', '{query}')
    ]
)
chain = prompt | llm_with_tools
tool_call_msg = chain.invoke('삼성SDS의 GPUaaS 서비스가 뭐야?')

tool_call_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_3Wm6eG64P7lhGPPT1oeigABQ', 'function': {'arguments': '{"query":"삼성SDS GPUaaS 서비스","max_results":5}', 'name': 'tavily_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 226, 'total_tokens': 327, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C7yAQMRU3mfRR6GCFd5grM8qIu1D9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--cfdcd444-639a-4593-8426-e1668f2d2803-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '삼성SDS GPUaaS 서비스', 'max_results': 5}, 'id': 'call_3Wm6eG64P7lhGPPT1oeigABQ', 'type': 'tool_call'}], usage_metadata={'input_tokens': 226, 'output_tokens': 

`tool_calls`에 포함된 name을 활용하면 툴 결과를 전달할 수 있습니다.   
`name` 값은 문자열이므로, dictionary를 통해 연결합니다.

In [22]:
tool_call_msg.tool_calls

[{'name': 'tavily_search',
  'args': {'query': '삼성SDS GPUaaS 서비스', 'max_results': 5},
  'id': 'call_3Wm6eG64P7lhGPPT1oeigABQ',
  'type': 'tool_call'}]

Tool에 tool_call을 입력해 invoke를 수행합니다.

In [24]:
tool_list = {'tavily_search': tavily_search}
tool_exec = tool_list[tool_call_msg.tool_calls[0]['name']]
tool_exec

StructuredTool(name='tavily_search', description='Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요.', args_schema=<class 'langchain_core.utils.pydantic.tavily_search'>, func=<function tavily_search at 0x000001D04AAF7E20>)

In [26]:
tool_msg = tool_exec.invoke(tool_call_msg.tool_calls[0])
tool_msg

ToolMessage(content='URL: https://www.youtube.com/watch?v=EQt26isav4s\n Title:[고객사례] 업스테이지의 든든한 엔진룸! GPU as a Service 삼성SDS ...\n Content:[고객사례] 업스테이지의 든든한 엔진룸! GPU as a Service │ 삼성SDS GPUaaS\n삼성SDS\n25700 subscribers\n19 likes\n59493 views\n27 Mar 2025\n고성능 GPU를 온프레미스로 사용하기엔 비용과 확장성 측면에서 어려운 부분이 있는데, GPUaaS(GPU as a Service)로 이러한 페인포인트를 해결할 수 있습니다. \n\n오픈 LLM 리더보드 세계 1위 기업 업스테이지(Upstage)는 삼성 클라우드 플랫폼 GPUaaS를 사용하여 AI 솔루션 개발에 필수적인 고성능 GPU 수급 문제를 해결하고, 개발 진행 속도에 박차를 가할 수 있었습니다.\n\n본 영상을 통해 업스테이지가 삼성 클라우드 플랫폼 GPUaaS를 선택한 이유와 도입 성공 사례를 확인해보세요!\n\n\n⚡️삼성SDS GPUaaS가 좋은 이유\n1) NVIDIA H100 등 국내 최대 GPU 보유량 기반, 국내 최대 규모 GPU 서비스 운영\n2) 도입 상담부터 구축, 원활한 운영까지 컨설턴트의 End-to-End 1:1 무상 기술지원\n3) 국내 리전 기반의 안정적 서비스 및 Facility 전문가의 24시간 관제 서비스\n\n💎GPUaaS 특별 프로모션 할인 혜택도 놓치지 마세요!\n1) 최초 사용 후 6개월 간 특별 할인 혜택 제공\n2) 고객의 목적과 환경에 맞춘 최적의 상품 구성 및 견적 제안\n➡️ https://www.samsungsds.com/kr/event/gpuaas-event1.html\n\n#gpuaas #gpuasaservice #aigpu #업스테이지 #Upstage #nvidiagpu #nvidiah100 #삼성SDS #삼성에스디에스 #삼성클라우드플랫폼 #Samsu

ToolMessage라는 새로운 형태의 메시지가 생성되었습니다.

HumanMessage, AIMessage(Tool Call), ToolMessage를 모두 전달합니다.

In [30]:
messages= []

messages.append(HumanMessage('삼성SDS의 GPUaaS 서비스가 뭐야?'))
# 질문이 들어있는 HumanMessage + 툴 요청이 들어있는 AIMessage + 툴 결과가 들어있는 ToolMessage
messages.append(tool_call_msg)
messages.append(tool_msg)



`Query-Tool Call-Tool`의 형식은 가장 기본적인 툴 사용 방법입니다.

In [31]:
result = llm_with_tools.invoke(messages)
print(result.content)

간단히 말하면, 삼성SDS의 GPUaaS는 “GPU를 클라우드에서 서비스 형태로 빌려 쓰는” 서비스입니다. 자체적으로 GPU 서버를 구축·운영하지 않아도 필요할 때 필요한 성능(GPU)을 즉시 할당받아 AI 모델 학습·추론, 대규모 연산, 그래픽 렌더링 등을 할 수 있게 해줍니다.

주요 특징(요약)
- 고성능 GPU 제공: NVIDIA H100 등 최신 고성능 GPU를 포함한 대규모 GPU 풀 보유(국내 최대 규모 운영을 표방).
- 즉시 사용·확장성: 필요할 때 즉시 인스턴스 생성·확장하여 학습/추론 작업 수행 가능.
- 매니지드 서비스: 도입 상담부터 구축·운영·모니터링까지 컨설턴트의 End-to-End 지원(1:1 기술지원, 24시간 관제).
- 국내 리전 기반: 국내 데이터센터에서 서비스되어 데이터 레지던시·응답속도 측면에서 이점.
- 요금·프로모션: 사용량 기반 요금(또는 계약형)이며, 신규 고객용 할인 프로모션 등을 제공하기도 함.

주요 활용 사례
- 대형 언어모델(LLM)·딥러닝 모델 학습 및 파인튜닝
- 대규모 추론(실시간/배치)
- 컴퓨터 비전, 시뮬레이션, 렌더링 등 고성능 연산 작업
- AI 연구·개발 환경(데브/테스트) 빠른 프로비저닝

장점
- 초기 투자(서버·공간·냉각 등)와 유지관리 비용 절감
- GPU 수급 문제(공급 지연) 해소, 빠른 확장
- 운영·보안·모니터링을 전문팀이 관리

고려할 점(단점/주의사항)
- 네트워크 대역폭·지연(latency)이 민감한 워크로드는 영향 받을 수 있음
- 장기적으로 매우 큰 용량을 지속 사용하면 온프레미스 대비 비용 차이 발생 가능
- 데이터 전송(특히 외부망·이전) 비용 및 규정 준수 확인 필요
- 특정 하드웨어(예: H100) 가용성은 시점에 따라 변동 가능

시작 방법(권장 절차)
1. 필요 GPU 스펙·예상 사용량(시간/인스턴스)을 정리
2. 삼성SDS 상담/견적 요청(온라인 문의 또는 이벤트 페이지 통해 신청)
3. 테스트(트라이얼 또는 PoC)로 성능·비용 검증
4. 운영 전 보안